In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-6"

In [ ]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else {
            "type": "text",
            "text": message,
            "cache_control": {
                "type": "ephemeral",
            },
        },        
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

def thinking_from_message(message):
    return "\n".join([block.thinking for block in message.content if block.type == "thinking"])


In [3]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [10]:
# TODO: Read image data, feed into Claude
import glob
import os

# Helper to read png files as bytes and create Claude image blocks
def get_claude_image_blocks_from_pngs(image_dir="./images"):
    image_blocks = []
    files = glob.glob(os.path.join(image_dir, "*.png"))
    for file_path in files:
        with open(file_path, "rb") as f:
            image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

        # Anthropic's Claude expects image content blocks of type "image"
        image_block = {
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": image_bytes,
            },
        }
        image_blocks.append(image_block)
    return image_blocks

def get_claude_pdf_blocks_from_pdfs(pdf_dir="./pdfs"):
    pdf_blocks = []
    files = glob.glob(os.path.join(pdf_dir, "*.pdf"))
    for file_path in files:
        with open(file_path, "rb") as f:
            pdf_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
    
        pdf_blocks.append({
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": pdf_bytes,
            },
            "title": os.path.basename(file_path),
            "citations": {
                "enabled": True,
            },
        })
    return pdf_blocks
# Compose the message array with image blocks for Claude
messages = []

# Add image blocks for all PNGs in ./images
# image_blocks = get_claude_image_blocks_from_pngs("./images")
# user_message = [
#     *image_blocks,
#     {
#         "type": "text",
#         "text": prompt,
#     },
# ]
# Add the prompt
# add_user_message(messages, user_message)

# response = chat(messages, thinking=True)
# text = text_from_message(response)
# thinking = thinking_from_message(response)
# print(text)
# print(thinking)

pdf_blocks = get_claude_pdf_blocks_from_pdfs("./pdfs")
user_message = [
    *pdf_blocks,
    {
        "type": "text",
        "text": "How were earth atmospheres formed?",
    },
]

add_user_message(messages, user_message)
response = chat(messages, thinking=True)
print(response)
text = text_from_message(response)
thinking = thinking_from_message(response)
print(text)
print(thinking)



Message(id='msg_01N1vsPEzSXCPNDeJGneWCXd', content=[ThinkingBlock(signature='EvgBClsIDRgCKkBBbD1q76YywC53rCQoIBROscTYWiQTpgP32aaEyNf2UrVaiYDobU9eRkbqDJd+d8OtfYxN1yPWl5N5We3eN2nZMhFjbGF1ZGUtc29ubmV0LTQtNjgAEgzg+l1a5yNDESmZlk8aDFa4DxT6kSuH/FRpkSIwpK0iYS3SC0KioOfjuAu6CRtJq8tsem064RcDuanxm+jHVVfUArMYVd1Z0Sz3YTxfKkvBGY4BOjngBA/LsFa5DbnDv1jD+vGZF0ULKdQ92vXc4xU6Oi/pkidBS7QZGwCs1bF0Hbqp7TUqRbi6Lk9N0WhhbpGj6MxjCNUSGcoYAQ==', thinking="The user is asking about how Earth's atmosphere was formed.", type='thinking'), TextBlock(citations=None, text="Based on the document, here is how Earth's atmosphere was formed:\n\n", type='text'), TextBlock(citations=[CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n", document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start